# SBIR entities in SBA's defense-critical NAICS codes

**Status:** completed exploratory companion notebook; non-citable  
**Research questions:** A1, A2/B2, B3, D1, and E2/E3 in `docs/research-questions.md`  
**Decision this informs:** identify SBIR-linked outreach pools and defensible public-procurement transition cases  
**Analysis date:** 2026-09-10  
**Procurement data through:** 2026-07-03  
**Canonical computation:** `scripts/data/defense_critical_naics_sbir.py`

This notebook reads generated evidence artifacts. It does not recreate identity, SAM parsing, contract rollup, phase classification, or transition logic. A NAICS registration, any target-coded procurement, a positive post-Phase-II transaction not classified as Phase I/II, a qualifying target-origin award after a clean archive baseline, and literal first-observed target entry remain separate observables.

## Data contract

- **Population:** validated SBIR recipients with an exact normalized 12-character UEI.
- **Primary grain:** one exact-UEI entity; SAM evidence is registration-row grain, procurement evidence is UEI × award × NAICS grain, and the identity-review queue is UEI × normalized SBIR-name key × normalized contract-name key grain.
- **Keys:** canonical SBIR compound award key, exact UEI, USAspending transaction unique ID, UEI-scoped generated award key, and the three-part identity-review key.
- **NAICS rule:** exact equality to the ten six-digit codes in SBA's September 10, 2026 announcement; no inferred or prefix matches.
- **Exclusions:** invalid/missing UEIs, IDVs, same-day post-index events, Phase-I/II-classified follow-on actions for the loose transition estimand, and nonqualifying target-origin awards for the clean-baseline estimands.
- **Clean baseline:** three years of procurement-archive calendar coverage before the Phase II index and no target-coded action before or on that date. Coverage is not evidence of continuous observation or firm activity.
- **SAM caveat:** registration evidence comes from one public monthly extract. NUL-quoted free text containing literal pipes required deterministic parser repair; the fields are descriptive extract values, not an eligibility adjudication or historical registry.
- **Identity review:** low name similarity creates a review candidate; it is not an identity conclusion. Generated candidate annotations come from the tracked source `data/reference/defense_critical_naics_identity_crosswalk.csv`, and candidate keys absent from that file remain unreviewed.
- **Missingness:** no public evidence is not negative evidence; private sales, subawards, classified work, pre-FY2009 procurement, historical SAM snapshots, and some successor identities are absent.
- **Output status:** exploratory and non-citable. The manifest records source and output hashes but this is not a frozen evidence-tier study.

In [ ]:
import json
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPORT_DIR = REPO_ROOT / "data" / "reports" / "defense_critical_naics"
IDENTITY_CROSSWALK = (
    REPO_ROOT / "data" / "reference" / "defense_critical_naics_identity_crosswalk.csv"
)
RANDOM_SEED = 20260910

In [ ]:
ARTIFACTS = {
    "manifest": REPORT_DIR / "manifest.json",
    "summary": REPORT_DIR / "summary.json",
    "firm evidence": REPORT_DIR / "firm_evidence.csv",
    "by code": REPORT_DIR / "by_code.csv",
    "horizons": REPORT_DIR / "horizons.csv",
    "threshold sensitivity": REPORT_DIR / "transition_threshold_sensitivity.csv",
    "identity review candidates": REPORT_DIR / "identity_review_candidates.csv",
    "clean-baseline entrants": REPORT_DIR / "transitioners.csv",
    "literal first-observed entrants": REPORT_DIR / "literal_first_entry_transitioners.csv",
    "sustained clean-baseline entrants": REPORT_DIR / "sustained_transitioners.csv",
}
artifact_status = pd.DataFrame(
    [
        {"artifact": name, "path": path.relative_to(REPO_ROOT), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)
artifact_status

In [ ]:
missing = artifact_status.loc[~artifact_status["exists"], "artifact"].tolist()
if missing:
    raise FileNotFoundError(
        f"Missing generated artifacts: {missing}. Run scripts/data/defense_critical_naics_sbir.py."
    )

manifest = json.loads(ARTIFACTS["manifest"].read_text())
summary = json.loads(ARTIFACTS["summary"].read_text())
firms = pd.read_csv(ARTIFACTS["firm evidence"], low_memory=False)
by_code = pd.read_csv(ARTIFACTS["by code"])
horizons = pd.read_csv(ARTIFACTS["horizons"])
thresholds = pd.read_csv(ARTIFACTS["threshold sensitivity"])
identity_candidates = pd.read_csv(
    ARTIFACTS["identity review candidates"], low_memory=False
)
clean_baseline = pd.read_csv(ARTIFACTS["clean-baseline entrants"], low_memory=False)
literal_entry = pd.read_csv(
    ARTIFACTS["literal first-observed entrants"], low_memory=False
)
sustained = pd.read_csv(
    ARTIFACTS["sustained clean-baseline entrants"], low_memory=False
)

pd.Series(
    {
        "epistemic_tier": manifest["epistemic_tier"],
        "citable": manifest["citable"],
        "analysis_date": manifest["as_of_date"],
        "procurement_data_through": summary["procurement_data_through"],
        "exact_uei_entities": len(firms),
        "post_phase_ii_positive_transaction_not_classified_phase_i_ii": summary[
            "entities_with_post_phase_ii_positive_transaction_not_classified_phase_i_ii"
        ],
        "clean_pre_index_qualifying_target_origin_award": len(clean_baseline),
        "literal_first_observed_target_entry": len(literal_entry),
        "sustained_clean_baseline_as_of_cutoff": len(sustained),
    },
    name="value",
)

## Registration and procurement overlap

SAM registration is a declaration in the public monthly extract; contract NAICS is an acquisition classification. Their overlap is informative, but neither is a universal test of company operations. The SAM snapshot cannot establish historical registration state, and its current 8(a) indicator is not an eligibility determination.

In [ ]:
n_total = summary["exact_uei_sbir_entities"]
n_sam = summary["entities_with_active_sam_target_code"]
n_procurement = summary["entities_with_any_target_prime_action"]
n_both = summary["overlap_active_sam_and_any_target_prime"]
overlap = pd.DataFrame(
    {
        "entities": [n_sam - n_both, n_procurement - n_both, n_both, n_sam + n_procurement - n_both],
    },
    index=["active SAM only", "procurement only", "both", "union"],
)
overlap["share_of_exact_uei_cohort"] = overlap["entities"] / n_total
overlap.style.format({"share_of_exact_uei_cohort": "{:.2%}"})

## Code-specific footprint

Entity columns are non-additive across rows. The HHI is computed only among matched SBIR entities' target-coded gross positive obligations, not the full market or industrial base.

In [ ]:
code_view = by_code[
    [
        "naics_code",
        "title",
        "active_sam_any_code_entities",
        "active_sam_primary_code_entities",
        "entities_with_any_target_prime_action",
        "clean_pre_index_qualifying_origin_entities",
        "literal_first_observed_entry_entities",
        "gross_positive_target_obligations",
        "sbir_cohort_gross_obligation_hhi_0_10000",
    ]
].sort_values("clean_pre_index_qualifying_origin_entities", ascending=False)
code_view.style.format(
    {
        "gross_positive_target_obligations": "${:,.0f}",
        "sbir_cohort_gross_obligation_hhi_0_10000": "{:,.0f}",
    }
)

## Fixed-horizon transition estimates

Each horizon uses a different fully observed risk set. `post_phase_ii_positive_transaction_not_classified_phase_i_ii` is the loose transaction estimand. The two clean-baseline estimands require three years of procurement-archive coverage before the Phase II index and no target-coded action before or on that date; this is calendar coverage, not proof that the firm was active or continuously observed. `clean_pre_index_qualifying_target_origin_award_3y_coverage` counts a qualifying first post-index target-origin award (205 in the overall cohort), while `literal_first_observed_target_entry_3y_coverage` additionally requires that award date to equal the entity's earliest target-coded action date (203 overall).

In [ ]:
horizon_estimands = [
    "post_phase_ii_positive_transaction_not_classified_phase_i_ii",
    "clean_pre_index_qualifying_target_origin_award_3y_coverage",
    "literal_first_observed_target_entry_3y_coverage",
]
horizon_view = horizons.loc[horizons["estimand"].isin(horizon_estimands)].assign(
    estimate=lambda frame: frame.apply(
        lambda row: (
            f"{int(row.observed_events):,} / "
            f"{int(row.eligible_exact_uei_phase_ii_entities):,} "
            f"({row.event_rate:.2%})"
        ),
        axis=1,
    )
).pivot(index="horizon_years", columns="estimand", values="estimate")
horizon_view = horizon_view.reindex(columns=horizon_estimands)
horizon_view

## Event-strength sensitivity

The first-date award can be economically small. For each estimand, this table thresholds the largest individual award among qualifying target-origin awards tied on the first event date. The amount is that award's gross-positive obligations over its observed lifetime through the procurement cutoff, including later modifications; it is not the obligation amount posted on the first date.

In [ ]:
thresholds.style.format(
    {
        "minimum_first_date_award_lifetime_gross_positive_obligations": "${:,.0f}",
        "median_latency_years": "{:.2f}",
    }
)

## Illustrative sustained clean-baseline entrants

This is an audit-oriented shortlist, not a causal ranking. Sustained status is an as-of-cutoff flag requiring at least two qualifying target-origin awards across at least two fiscal years; entrants have unequal follow-up, so the flag is not a common-horizon outcome and its absence is not a comparable failure. `target_gross_positive_obligations` includes all observed target-coded actions through the procurement cutoff.

In [ ]:
entrant_view = sustained[
    [
        "display_name",
        "firm_uei",
        "first_post_phase_ii_target_origin_award_date",
        "first_post_phase_ii_target_origin_award_codes",
        "clean_pre_index_latency_days",
        "first_post_phase_ii_target_origin_award_maximum_award_gross_positive_obligations",
        "target_gross_positive_obligations",
        "high_name_continuity",
    ]
].head(20)
entrant_view.style.format(
    {
        "clean_pre_index_latency_days": "{:,.0f}",
        "first_post_phase_ii_target_origin_award_maximum_award_gross_positive_obligations": "${:,.0f}",
        "target_gross_positive_obligations": "${:,.0f}",
    }
)

## Identity and acquisition diagnostic

Low name similarity does not invalidate an exact-UEI transaction. It is only a deterministic trigger for reviewing whether entity history makes the original SBIR label a poor description of the later contractor; by itself it does not establish an acquisition, name change, successor relationship, or federal contract novation.

In [ ]:
procurement_firms = firms.loc[firms["has_any_target_prime_action"].fillna(False)].copy()
identity_diagnostic = (
    procurement_firms.groupby("high_name_continuity", dropna=False)
    .agg(
        entities=("firm_uei", "nunique"),
        gross_positive_obligations=("target_gross_positive_obligations", "sum"),
    )
    .assign(
        dollar_share=lambda frame: (
            frame["gross_positive_obligations"] / frame["gross_positive_obligations"].sum()
        )
    )
)
identity_diagnostic.style.format(
    {"gross_positive_obligations": "${:,.0f}", "dollar_share": "{:.2%}"}
)

## Identity-review queue

`identity_review_candidates.csv` is a generated review queue, not a set of acquisition findings. It merges keyed annotations from the tracked source `data/reference/defense_critical_naics_identity_crosswalk.csv`. `reviewed_supported` means a matching crosswalk row documents a corporate relation; `reviewed_unresolved` means a matching review exists but the corporate relation remains unknown; and `unreviewed` means no matching crosswalk row exists. Missing review therefore remains unreviewed rather than becoming an unresolved or negative finding.

The corporate and contract questions remain separate. Evidence of an acquisition, merger, or name change does not prove that any federal contract was novated. `contract_relationship` and `attribution_treatment` preserve that distinction; for example, `not_established` records that the review did not establish the contract relationship even when `corporate_relation` is supported.

In [ ]:
review_states = ["reviewed_supported", "reviewed_unresolved", "unreviewed"]
priority_identity_review = identity_candidates.loc[
    identity_candidates["priority_review_tranche"].fillna(False)
].copy()
identity_review_counts = pd.Series(
    {
        "candidate_rows": len(identity_candidates),
        "candidate_entities": identity_candidates["firm_uei"].nunique(),
        "priority_tranche_rows": len(priority_identity_review),
        "priority_tranche_entities": priority_identity_review["firm_uei"].nunique(),
        "reviewed_supported_rows": identity_candidates["review_state"]
        .eq("reviewed_supported")
        .sum(),
        "reviewed_unresolved_rows": identity_candidates["review_state"]
        .eq("reviewed_unresolved")
        .sum(),
        "unreviewed_rows": identity_candidates["review_state"].eq("unreviewed").sum(),
        "tracked_review_source": str(IDENTITY_CROSSWALK.relative_to(REPO_ROOT)),
    },
    name="value",
)
identity_review_counts

In [ ]:
identity_review_status = (
    identity_candidates.groupby("review_state", dropna=False)
    .agg(
        candidate_rows=("firm_uei", "size"),
        candidate_entities=("firm_uei", "nunique"),
        gross_positive_obligations=("target_gross_positive_obligations", "sum"),
    )
    .reindex(review_states, fill_value=0)
)
identity_review_status["share_of_candidate_gross_positive_obligations"] = (
    identity_review_status["gross_positive_obligations"]
    / identity_review_status["gross_positive_obligations"].sum()
)
identity_review_status.style.format(
    {
        "gross_positive_obligations": "${:,.0f}",
        "share_of_candidate_gross_positive_obligations": "{:.2%}",
    }
)

### Priority 14-case tranche

The current priority tranche combines the ten highest-dollar flagged entities with every clean-pre-index entity in the low-similarity queue. The table shows the review state and keeps the documented corporate relation separate from the federal-contract relationship and attribution treatment.

In [ ]:
priority_identity_view = priority_identity_review[
    [
        "firm_uei",
        "sbir_display_name",
        "contract_recipient_name",
        "candidate_entity_dollar_rank",
        "priority_reason",
        "review_state",
        "corporate_relation",
        "contract_relationship",
        "attribution_treatment",
    ]
].sort_values(["candidate_entity_dollar_rank", "firm_uei"])
priority_identity_view

## Interpretation log

| Observation | Defensible statement | Do not infer |
|---|---|---|
| Active SAM target code | Current registration declares a target industry or capability | Verified production, sales, or eligibility |
| Target-coded prime action | The federal acquisition was coded to a target NAICS | The code is the firm's primary industry |
| Positive post-Phase-II transaction not classified as Phase I/II | A positive target-coded transaction occurs after the Phase II index and may modify an existing award | SBIR caused it, it is commercial work, or it is statutory Phase III |
| Clean-baseline qualifying target-origin award | A qualifying target-origin award follows three years of archive coverage with no earlier observed target action | Continuous firm observation or true first industry entry |
| Literal first-observed target entry | The clean-baseline award date is also the earliest target-coded action date in the archive | No earlier private, subcontract, classified, or pre-FY2009 activity |
| Low SBIR-to-contract name similarity | The identity pair enters a manual-review queue | Acquisition, name change, successor status, novation, or an invalid exact-UEI match |
| Reviewed corporate relation | The tracked crosswalk documents the stated corporate event | Federal contract novation or automatic attribution of later dollars to the original SBIR firm |
| High cohort HHI | Dollars are concentrated among matched SBIR entities | Market concentration or a physical supply bottleneck |

Before external citation, promote the work to the evidence tier with a frozen specification, blocking validation, reviewed identity crosswalk, and declared estimand.